# PY-06 | Διερευνητική Ανάλυση Δεδομένων (EDA) και Οπτικοποίηση

Σε αυτό το μάθημα συνεχίζουμε **ακριβώς από το αποτέλεσμα του PY-05**.

Θα χρησιμοποιήσουμε το πραγματικό processed αρχείο:

`data/processed/elstat_unemployment_2021_municipalities.csv`

και θα μάθουμε πώς να εξερευνούμε ένα dataset πριν προχωρήσουμε σε πιο σύνθετη ανάλυση.

> **Βασική ιδέα:**  
> Το EDA δεν είναι απλώς «φτιάχνω γραφήματα». Είναι η διαδικασία με την οποία προσπαθούμε να καταλάβουμε τη δομή, τις κατανομές, τις ακραίες τιμές και τις σχέσεις μέσα στα δεδομένα μας.

## 1. Στόχοι του μαθήματος

Στο τέλος του PY-06 θα μπορούμε να:

- εξετάζουμε γρήγορα τη δομή ενός dataset,
- χρησιμοποιούμε περιγραφικά στατιστικά,
- συγκρίνουμε mean και median,
- μελετάμε κατανομές με histograms,
- χρησιμοποιούμε boxplots και IQR για πιθανές ακραίες τιμές,
- ξεχωρίζουμε **counts** από **rates**,
- δημιουργούμε scatterplots,
- εξετάζουμε Pearson correlations,
- αναγνωρίζουμε skewed μεταβλητές,
- εφαρμόζουμε έναν απλό `log1p()` transformation,
- θυμόμαστε ότι **outlier ≠ error** και **correlation ≠ causation**.

## 2. Imports και path του dataset

Η σειρά υποθέτει **system Python**.

Αν λείπει κάποια βιβλιοθήκη:

```bash
python -m pip install pandas numpy matplotlib
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path(
    "data/processed/elstat_unemployment_2021_municipalities.csv"
)

## 3. Φόρτωση του processed dataset

Δεν ξανακάνουμε το cleaning του PY-05.

```text
PY-05
raw ELSTAT Excel
        ↓
cleaning + validation + merge
        ↓
processed CSV

PY-06
processed CSV
        ↓
EDA + visualization
```

Αυτό είναι σημαντικό: σε ένα πραγματικό data pipeline, το output ενός βήματος μπορεί να γίνει το input του επόμενου.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Δεν βρέθηκε το processed CSV. "
        "Τρέξε πρώτα το PY-05 ή έλεγξε το DATA_PATH."
    )

df = pd.read_csv(
    DATA_PATH,
    dtype={"geo_code": "string"}
)

df.head()

## 4. Πρώτη επιθεώρηση

Πριν δημιουργήσουμε οποιοδήποτε γράφημα, ελέγχουμε:

- πόσες γραμμές και στήλες έχουμε,
- τα column names,
- τα data types,
- missing values,
- duplicate geographic codes.

Με το συγκεκριμένο αρχείο περιμένουμε **333 municipal-level records**. Το dataset περιλαμβάνει και το **Άγιο Όρος (Αυτοδιοίκητο)**, κάτι που θα αποδειχθεί ενδιαφέρον όταν εξετάσουμε outliers.

In [ ]:
print("Shape:", df.shape)
print("Duplicate geo codes:", df["geo_code"].duplicated().sum())
print("Missing values:", df.isna().sum().sum())

df.info()

In [ ]:
df.columns.tolist()

## 5. Επιλογή μεταβλητών για το EDA

Δεν χρειάζεται να εξετάζουμε και τις 21 στήλες ταυτόχρονα.

Θα χρησιμοποιήσουμε ένα μικρό, ερμηνεύσιμο subset.

In [ ]:
eda_columns = [
    "population_total",
    "economically_active",
    "unemployed",
    "unemployment_rate",
    "edu_tertiary_pct",
    "edu_upper_secondary_pct",
    "edu_primary_pct",
]

eda = df[
    ["geo_code", "municipality", *eda_columns]
].copy()

eda.head()

## 6. Περιγραφικά στατιστικά

Η `.describe()` μας δίνει μια πρώτη συνοπτική εικόνα:

- `count`
- `mean`
- `std`
- `min`
- quartiles
- `max`

Η `.T` κάνει transpose τον πίνακα ώστε οι μεταβλητές να εμφανίζονται ως γραμμές.

In [ ]:
eda[eda_columns].describe().T

### Mean, median και skewness

Σε μια περίπου συμμετρική κατανομή, mean και median τείνουν να είναι σχετικά κοντά.

Σε έντονα skewed δεδομένα μπορεί να απέχουν πολύ.

Στο πραγματικό dataset θα δούμε μια έντονη αντίθεση:

- το `unemployment_rate` είναι σχετικά συμμετρικό,
- τα absolute counts, όπως `population_total` και `unemployed`, είναι έντονα right-skewed.

In [ ]:
summary = pd.DataFrame({
    "mean": eda[eda_columns].mean(),
    "median": eda[eda_columns].median(),
    "std": eda[eda_columns].std(),
    "skewness": eda[eda_columns].skew(),
})

summary.round(2)

### Τι βλέπουμε στα πραγματικά δεδομένα;

Το `population_total` έχει πολύ μεγάλη positive skewness, επειδή λίγοι πολύ μεγάλοι Δήμοι βρίσκονται πολύ μακριά από την πλειονότητα.

Αντίθετα, το `unemployment_rate` έχει mean και median αρκετά κοντά και πολύ μικρότερη skewness.

Αυτό είναι ένα καλό παράδειγμα του γιατί **count και rate συμπεριφέρονται διαφορετικά στατιστικά**.

## 7. Η κατανομή του ποσοστού ανεργίας

Ένα histogram χωρίζει τις τιμές σε διαστήματα (`bins`) και δείχνει πόσες παρατηρήσεις πέφτουν σε κάθε διάστημα.

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    eda["unemployment_rate"].dropna(),
    bins=20
)

plt.xlabel("Unemployment rate (%)")
plt.ylabel("Number of municipalities")
plt.title("Distribution of municipal unemployment rate")

plt.show()

### Τι κοιτάμε σε ένα histogram;

Ρωτάμε:

- Πού συγκεντρώνονται οι περισσότερες τιμές;
- Είναι η κατανομή περίπου συμμετρική;
- Υπάρχει μεγάλη ουρά;
- Υπάρχουν απομονωμένες παρατηρήσεις;

Στο dataset μας το unemployment rate είναι πολύ λιγότερο skewed από τα population counts.

## 8. Boxplot και πιθανές ακραίες τιμές

Το boxplot συνοψίζει την κατανομή χρησιμοποιώντας quartiles.

Θα χρησιμοποιήσουμε επίσης τον **IQR**:

\[
IQR = Q_3 - Q_1
\]

Ένας συνηθισμένος διερευνητικός κανόνας θεωρεί ως πιθανές ακραίες τιμές όσες βρίσκονται έξω από:

\[
Q_1 - 1.5 \times IQR
\]

και

\[
Q_3 + 1.5 \times IQR
\]

Αυτό είναι **κανόνας εντοπισμού**, όχι απόδειξη ότι μια παρατήρηση πρέπει να διαγραφεί.

In [ ]:
plt.figure(figsize=(8, 3))

plt.boxplot(
    eda["unemployment_rate"].dropna(),
    vert=False
)

plt.xlabel("Unemployment rate (%)")
plt.title("Municipal unemployment rate")

plt.show()

In [ ]:
q1 = eda["unemployment_rate"].quantile(0.25)
q3 = eda["unemployment_rate"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

In [ ]:
possible_outliers = eda[
    (eda["unemployment_rate"] < lower_bound)
    | (eda["unemployment_rate"] > upper_bound)
]

possible_outliers[
    [
        "municipality",
        "unemployed",
        "economically_active",
        "unemployment_rate",
    ]
].sort_values("unemployment_rate")

### Τι μας δείχνει το πραγματικό αποτέλεσμα;

Ο κανόνας `1.5 × IQR` εντοπίζει λίγες παρατηρήσεις εκτός των ορίων.

Η πιο χαρακτηριστική είναι το **Άγιο Όρος**, με εξαιρετικά χαμηλό unemployment rate. Αυτό δεν σημαίνει ότι η γραμμή είναι λάθος. Πρόκειται για μια πολύ ιδιαίτερη γεωγραφική και κοινωνική περίπτωση.

Άρα:

> **Outlier ≠ error**

Το EDA μάς λέει «κοίταξε αυτή την παρατήρηση πιο προσεκτικά», όχι «διέγραψέ την».

## 9. Counts και rates δεν απαντούν στην ίδια ερώτηση

Ένας μεγάλος Δήμος μπορεί να έχει πολλούς ανέργους επειδή έχει μεγάλο πληθυσμό.

Αυτό δεν σημαίνει απαραίτητα ότι έχει και υψηλό **ποσοστό ανεργίας**.

In [ ]:
top_unemployed = eda.nlargest(
    10,
    "unemployed"
)

top_rate = eda.nlargest(
    10,
    "unemployment_rate"
)

top_unemployed[
    ["municipality", "unemployed", "unemployment_rate"]
]

In [ ]:
top_rate[
    ["municipality", "unemployed", "unemployment_rate"]
]

Οι δύο κατατάξεις είναι πολύ διαφορετικές.

Οι μεγάλοι αστικοί Δήμοι κυριαρχούν στα **absolute counts**, ενώ οι υψηλότερες τιμές του **rate** εμφανίζονται και σε πολύ μικρότερους Δήμους.

In [ ]:
plot_data = (
    top_rate[
        ["municipality", "unemployment_rate"]
    ]
    .sort_values("unemployment_rate")
)

plt.figure(figsize=(8, 5))

plt.barh(
    plot_data["municipality"],
    plot_data["unemployment_rate"]
)

plt.xlabel("Unemployment rate (%)")
plt.title("10 municipalities with the highest unemployment rate")

plt.show()

## 10. Scatterplot: population και unemployed

Αρχικά συγκρίνουμε δύο absolute counts:

- συνολικό πληθυσμό
- αριθμό ανέργων

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    eda["population_total"],
    eda["unemployed"]
)

plt.xlabel("Total population")
plt.ylabel("Unemployed")
plt.title("Population and number of unemployed")

plt.show()

Στο πραγματικό dataset η σχέση είναι πολύ ισχυρή και θετική.

Αυτό δεν είναι παράξενο: ένας Δήμος με πολύ μεγαλύτερο πληθυσμό έχει τη δυνατότητα να έχει και πολύ περισσότερους ανέργους σε **απόλυτο αριθμό**.

Γι' αυτό τα counts συχνά αντανακλούν σε μεγάλο βαθμό το μέγεθος της περιοχής.

## 11. Scatterplots με κοινωνικούς δείκτες

Τώρα συγκρίνουμε το `unemployment_rate` με δύο μεταβλητές που περιγράφουν τη **σύνθεση των ανέργων** ανά εκπαιδευτική κατηγορία.

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    eda["edu_tertiary_pct"],
    eda["unemployment_rate"]
)

plt.xlabel("Tertiary education among unemployed (%)")
plt.ylabel("Unemployment rate (%)")
plt.title("Tertiary education composition and unemployment rate")

plt.show()

Παρατηρούμε μια **αρνητική association**: Δήμοι όπου μεγαλύτερο ποσοστό των ανέργων ανήκει στην τριτοβάθμια εκπαιδευτική κατηγορία τείνουν να έχουν χαμηλότερο συνολικό unemployment rate.

Αλλά προσέχουμε τον παρονομαστή:

`edu_tertiary_pct` σημαίνει:

> ποιο ποσοστό **όλων των ανέργων του Δήμου** ανήκει στην τριτοβάθμια εκπαιδευτική κατηγορία.

Δεν σημαίνει:

> ποιο ποσοστό των ατόμων με τριτοβάθμια εκπαίδευση είναι άνεργο.

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    eda["edu_primary_pct"],
    eda["unemployment_rate"]
)

plt.xlabel("Primary education among unemployed (%)")
plt.ylabel("Unemployment rate (%)")
plt.title("Primary education composition and unemployment rate")

plt.show()

Στο συγκεκριμένο dataset αυτή η σχέση είναι θετική και οπτικά πιο έντονη από τη σχέση με το `edu_tertiary_pct`.

Και πάλι, όμως, εξετάζουμε **association**, όχι αιτιότητα.

## 12. Συσχέτιση

Ο Pearson correlation coefficient παίρνει τιμές από `-1` έως `1`.

- κοντά στο `1` → ισχυρή θετική γραμμική σχέση,
- κοντά στο `-1` → ισχυρή αρνητική γραμμική σχέση,
- κοντά στο `0` → ασθενής γραμμική σχέση.

**Correlation ≠ causation.**

In [ ]:
correlation_columns = [
    "population_total",
    "economically_active",
    "unemployed",
    "unemployment_rate",
    "edu_tertiary_pct",
    "edu_upper_secondary_pct",
    "edu_primary_pct",
]

correlations = (
    eda[correlation_columns]
    .corr()
    .round(2)
)

correlations

### Μερικά πραγματικά patterns

Στο dataset μας:

- `population_total` και `unemployed` έχουν σχεδόν τέλεια θετική correlation,
- `population_total` και `unemployment_rate` έχουν σχεδόν μηδενική γραμμική correlation,
- `edu_tertiary_pct` και `unemployment_rate` έχουν μέτρια αρνητική association,
- `edu_primary_pct` και `unemployment_rate` έχουν μέτρια θετική association.

Αυτό δείχνει γιατί μια correlation matrix μπορεί να αποκαλύψει πολύ διαφορετικά είδη σχέσεων.

### Πού υπάρχει πιθανή redundancy;

Μεγάλες absolute correlations μεταξύ μεταβλητών μπορεί να σημαίνουν ότι κάποιες μεταφέρουν πολύ παρόμοια πληροφορία.

Για παράδειγμα, στο συγκεκριμένο dataset:

- `population_total`,
- `economically_active`,
- `unemployed`

είναι πολύ ισχυρά συσχετισμένα.

Δεν θα λύσουμε το πρόβλημα της redundancy εδώ.

Θα επιστρέψουμε συστηματικά σε αυτό στο επεισόδιο **PCA**, με μεγαλύτερο σύνολο υποψήφιων μεταβλητών.

## 13. Skewness και log transformation

Ο `population_total` είναι εξαιρετικά right-skewed στο πραγματικό dataset.

Θα χρησιμοποιήσουμε:

```python
np.log1p(x)
```

που υπολογίζει:

\[
\log(1 + x)
\]

Το `+1` επιτρέπει τον μετασχηματισμό και όταν μια τιμή είναι `0`.

In [ ]:
print(
    "Population skewness:",
    eda["population_total"].skew()
)

eda["log_population_total"] = np.log1p(
    eda["population_total"]
)

print(
    "Log population skewness:",
    eda["log_population_total"].skew()
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    eda["population_total"].dropna(),
    bins=20
)

plt.xlabel("Population")
plt.ylabel("Number of municipalities")
plt.title("Population before log transformation")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    eda["log_population_total"].dropna(),
    bins=20
)

plt.xlabel("log(1 + population)")
plt.ylabel("Number of municipalities")
plt.title("Population after log transformation")

plt.show()

Στο πραγματικό dataset η skewness του population πέφτει δραστικά μετά τον log transformation.

Αυτό δεν σημαίνει ότι η transformed κατανομή γίνεται «τέλεια normal».

> Ο log transformation αλλάζει την κλίμακα και συμπιέζει τις πολύ μεγάλες τιμές. Δεν είναι αυτόματη «διόρθωση» των δεδομένων.

## 14. Μέση εκπαιδευτική σύνθεση των ανέργων

Μπορούμε να συνοψίσουμε τη μέση σύνθεση των ανέργων για μερικές εκπαιδευτικές κατηγορίες.

Θυμόμαστε ότι αυτά είναι **composition percentages**.

In [ ]:
education_pct_columns = [
    "edu_tertiary_pct",
    "edu_upper_secondary_pct",
    "edu_primary_pct",
]

education_means = (
    eda[education_pct_columns]
    .mean()
    .sort_values()
)

education_means

In [ ]:
plt.figure(figsize=(8, 4))

plt.barh(
    education_means.index,
    education_means.values
)

plt.xlabel("Mean share among unemployed (%)")
plt.title("Average educational composition of unemployed")

plt.show()

## Καλή πρακτική για τη διάθεση επίσημων στατιστικών

Το EDA δείχνει γιατί δεν αρκεί ένας πάροχος να δημοσιεύει μόνο έναν τελικό δείκτη.

Για σωστή διερεύνηση είναι χρήσιμο να παρέχονται μαζί:

- το **raw count**,
- ο παρονομαστής,
- η μονάδα μέτρησης,
- ο ακριβής ορισμός της μεταβλητής,
- το γεωγραφικό επίπεδο,
- metadata,
- version / revision information.

Για παράδειγμα, το `unemployment_rate` είναι πολύ πιο ερμηνεύσιμο επειδή έχουμε ταυτόχρονα τους `unemployed` και `economically_active`.

Σε machine-readable datasets ή APIs, αυτές οι πληροφορίες μπορούν να συνοδεύουν τα δεδομένα με συνεπή τρόπο και να κάνουν την ανάλυση πιο αναπαραγώγιμη.

## 15. Μικρό EDA summary table

Συγκεντρώνουμε βασικά στατιστικά για τις μεταβλητές του μαθήματος.

In [ ]:
eda_summary = pd.DataFrame({
    "mean": eda[eda_columns].mean(),
    "median": eda[eda_columns].median(),
    "std": eda[eda_columns].std(),
    "min": eda[eda_columns].min(),
    "max": eda[eda_columns].max(),
    "skewness": eda[eda_columns].skew(),
}).round(2)

eda_summary

## 16. Προαιρετικό export

Το κύριο processed dataset εξακολουθεί να είναι το αρχείο του PY-05.

Για λόγους διδασκαλίας μπορούμε να αποθηκεύσουμε και μια έκδοση του EDA subset που περιλαμβάνει τη νέα `log_population_total`.

In [ ]:
OUTPUT_PATH = Path(
    "data/processed/elstat_unemployment_2021_eda.csv"
)

eda.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

## 17. Ασκήσεις

### Άσκηση 1
Βρες τους 10 Δήμους με το **χαμηλότερο** ποσοστό ανεργίας.

### Άσκηση 2
Υπολόγισε mean, median και skewness για τη μεταβλητή `unemployed`.

### Άσκηση 3
Δημιούργησε scatterplot μεταξύ:

- `economically_active`
- `unemployment_rate`

### Άσκηση 4
Υπολόγισε τον IQR για τη μεταβλητή `population_total` και εντόπισε τις πιθανές ακραίες τιμές.

### Άσκηση 5
Σύγκρινε τη συσχέτιση:

- `population_total` με `unemployed`
- `population_total` με `unemployment_rate`

Προσπάθησε να εξηγήσεις γιατί οι δύο σχέσεις είναι τόσο διαφορετικές.

In [ ]:
# Άσκηση 1

In [ ]:
# Άσκηση 2

In [ ]:
# Άσκηση 3

In [ ]:
# Άσκηση 4

In [ ]:
# Άσκηση 5

## 18. Σύνοψη

Στο PY-06 είδαμε ότι πριν εφαρμόσουμε πιο σύνθετες μεθόδους πρέπει πρώτα να γνωρίζουμε τα δεδομένα μας.

```text
inspect
   ↓
describe
   ↓
visualize distributions
   ↓
check possible outliers
   ↓
compare counts and rates
   ↓
explore relationships
   ↓
correlations
   ↓
check skewness
   ↓
consider transformations
```

Το πραγματικό ELSTAT dataset μάς έδωσε πολύ καλά παραδείγματα:

- έντονα skewed population counts,
- σχετικά συμμετρικό unemployment rate,
- πραγματικά outliers,
- πολύ ισχυρή correlation μεταξύ population και unemployed count,
- σχεδόν μηδενική σχέση population και unemployment rate,
- διαφορετικές associations μεταξύ unemployment rate και εκπαιδευτικής σύνθεσης.

Στο επόμενο επεισόδιο θα προσθέσουμε τη **γεωγραφική διάσταση** με GeoPandas.

## 19. Λύσεις ασκήσεων

Προσπάθησε πρώτα να λύσεις τις ασκήσεις του **Τμήματος 17** χωρίς να κοιτάξεις παρακάτω.

### Άσκηση 1 — 10 Δήμοι με το χαμηλότερο ποσοστό ανεργίας

In [ ]:
lowest_rate = eda.nsmallest(
    10,
    "unemployment_rate"
)

lowest_rate[
    ["municipality", "unemployment_rate"]
]

### Άσκηση 2 — Mean, median και skewness για `unemployed`

In [ ]:
unemployed_mean = eda["unemployed"].mean()
unemployed_median = eda["unemployed"].median()
unemployed_skewness = eda["unemployed"].skew()

print("Mean:", unemployed_mean)
print("Median:", unemployed_median)
print("Skewness:", unemployed_skewness)

### Άσκηση 3 — Scatterplot: economically active και unemployment rate

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(
    eda["economically_active"],
    eda["unemployment_rate"]
)

plt.xlabel("Economically active population")
plt.ylabel("Unemployment rate (%)")
plt.title("Economically active population and unemployment rate")

plt.show()

### Άσκηση 4 — IQR και possible outliers για `population_total`

In [ ]:
population_q1 = eda["population_total"].quantile(0.25)
population_q3 = eda["population_total"].quantile(0.75)

population_iqr = population_q3 - population_q1

population_lower = population_q1 - 1.5 * population_iqr
population_upper = population_q3 + 1.5 * population_iqr

print("Q1:", population_q1)
print("Q3:", population_q3)
print("IQR:", population_iqr)
print("Lower bound:", population_lower)
print("Upper bound:", population_upper)

In [ ]:
population_outliers = eda[
    (eda["population_total"] < population_lower)
    | (eda["population_total"] > population_upper)
]

population_outliers[
    ["municipality", "population_total"]
].sort_values("population_total")

### Άσκηση 5 — Δύο διαφορετικές συσχετίσεις

In [ ]:
corr_population_unemployed = eda[
    ["population_total", "unemployed"]
].corr().iloc[0, 1]

corr_population_rate = eda[
    ["population_total", "unemployment_rate"]
].corr().iloc[0, 1]

print(
    "Population vs unemployed:",
    corr_population_unemployed
)

print(
    "Population vs unemployment rate:",
    corr_population_rate
)

**Ερμηνεία:** στο πραγματικό dataset η πρώτη correlation είναι εξαιρετικά υψηλή, ενώ η δεύτερη είναι πολύ κοντά στο μηδέν.

Ο συνολικός πληθυσμός και ο αριθμός ανέργων είναι και τα δύο absolute counts και επηρεάζονται έντονα από το μέγεθος του Δήμου.

Το `unemployment_rate`, αντίθετα, είναι σχετικό μέγεθος:

\[
\frac{\text{unemployed}}{\text{economically active}} \times 100
\]

και δεν αυξάνεται απλώς επειδή ένας Δήμος είναι μεγαλύτερος.